In [ ]:
import pandas as pd
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from pathlib import Path
from socket import gethostname
import seaborn as sns

host = gethostname()

if host == 'gcer-a100':
    font_path = '/home/dhester/.fonts/InterVariable.ttf'
elif host == 'ABE-BC546S3':
    font_path = r"C:\Users\dh2306\AppData\Local\Microsoft\Windows\Fonts\Inter-VariableFont_opsz,wght.ttf"
elif host == 'MacBookPro':
    font_path = '/Users/dak/Library/Fonts/Inter-VariableFont_opsz,wght.ttf'
else:
    raise NotImplementedError("Unknown host. Please set the Inter font path for this machine.")

font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path)

# plt.style.use('bmh')
plt.style.use('seaborn-v0_8-colorblind')
# plt.style.use('fivethirtyeight')
# set grid background color to white
# plt.rcParams['axes.facecolor'] = 'white'
# use outline around axes for better visibility when multiple axes are present
# plt.rcParams['axes.edgecolor'] = 'black'
# plt.rcParams['axes.linewidth'] = 0.5


plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = prop.get_name()

In [ ]:
lc_df = pd.read_csv('../runs/s2flow_lc_sampling_steps_sweep/lc_summary_metrics_all.csv')

lc_df_reduced = lc_df.loc[lc_df['sampler'].isin(['s2', 'realesrgan', 'euler', 'naip']) & lc_df['steps'].isin([0, 1])]
lc_df_reduced_mean = lc_df_reduced.groupby(['model', 'sampler', 'steps']).mean(numeric_only=True).reset_index()
lc_df_reduced_std = lc_df_reduced.groupby(['model', 'sampler', 'steps']).std(numeric_only=True).reset_index()

fancy_sampler_names = {
    'euler': 'Euler',
    'ddpm': 'DDPM',
    'ddim': 'DDIM',
}
fancy_model_names = {
    'segformer': 'SegFormer',
    'deeplabv3plus': 'DeepLabV3+',
    'unet': 'U-Net'
}

lc_df_reduced_mean['model'] = lc_df_reduced_mean['model'].map(fancy_model_names)
lc_df_reduced_mean['model'] = pd.Categorical(
    lc_df_reduced_mean['model'],
    categories=['U-Net', 'DeepLabV3+', 'SegFormer'],
    ordered=True
)
lc_df_reduced_std['model'] = lc_df_reduced_std['model'].map(fancy_model_names)
lc_df_reduced_std['model'] = pd.Categorical(
    lc_df_reduced_std['model'],
    categories=['U-Net', 'DeepLabV3+', 'SegFormer'],
    ordered=True
)

lc_df_reduced_mean['sampler'] = lc_df_reduced_mean['sampler'].map(fancy_data_labels)
lc_df_reduced_mean['sampler'] = pd.Categorical(
    lc_df_reduced_mean['sampler'],
    categories=['Sentinel-2', 'Real-ESRGAN', r'Euler ($T=1$)', 'NAIP'],
    ordered=True
)
lc_df_reduced_std['sampler'] = lc_df_reduced_std['sampler'].map(fancy_data_labels)
lc_df_reduced_std['sampler'] = pd.Categorical(
    lc_df_reduced_std['sampler'],
    categories=['Sentinel-2', 'Real-ESRGAN', r'Euler ($T=1$)', 'NAIP'],
    ordered=True
)

mapping = {
    'model': 'Model',
    'sampler': 'Input Data Source',
    'macro_precision': "User's Accuracy (%)",
    'macro_recall': "Producer's Accuracy (%)",
    'macro_f1': 'F1 Score (%)',
    'overall_accuracy': 'Overall Accuracy (%)',
}
reverse_mapping = {v: k for k, v in mapping.items()}
lc_df_table = lc_df_reduced_mean.sort_values(['model', 'sampler'])[['model', 'sampler', 'macro_precision', 'macro_recall', 'macro_f1', 'overall_accuracy']]
lc_df_table = lc_df_table.rename(columns=mapping)

# convert to percent format
for col in ["User's Accuracy (%)", "Producer's Accuracy (%)", 'F1 Score (%)', 'Overall Accuracy (%)']:
    lc_df_table[col] = lc_df_table[col] * 100.0
    lc_df_table[col] = lc_df_table[col].round(2)
    lc_df_table[col] = lc_df_table[col].map(lambda x: f"{x:.2f}")
    
    # now add std
    std_col = lc_df_reduced_std.sort_values(['model', 'sampler'])[['model', 'sampler', reverse_mapping[col]]]
    std_col = std_col.rename(columns={reverse_mapping[col]: 'std'})
    std_col['std'] = std_col['std'] * 100.0
    std_col['std'] = std_col['std'].round(2)
    # make sure std_col has 2 decimal places
    std_col['std'] = std_col['std'].map(lambda x: f"{x:.2f}")
    
    # now combine
    lc_df_table[col] = lc_df_table.apply(
        lambda row: f"{row[col]} ± {std_col.loc[(std_col['model'] == row['Model']) & (std_col['sampler'] == row['Input Data Source']), 'std'].values[0]}",
        axis=1
    )

lc_df_table
# lc_df_table.to_csv('../runs/s2flow_lc_sampling_steps_sweep/sampling_lc_summary_table.csv', index=False)

In [ ]:

deltas = []
metrics = ['precision', 'recall', 'f1_score']

fancy_data_labels = {
    's2': 'Sentinel-2',
    'realesrgan': 'Real-ESRGAN',
    'euler': 'Euler ($T=1$)',
    'naip': 'NAIP',
}
fancy_metric_names = {
    'precision': 'User\'s Accuracy',
    'recall': 'Producer\'s Accuracy',
    'f1_score': 'F1 Score',
    # 'mean_iou': 'mIoU',
}

class_names = {
    1: 'Open Water',
    2: 'Herbaceous',
    3: 'Forest Canopy',
    4: 'Impervious',
    5: 'Barren Land',
}

fancy_model_names = {
    'segformer': 'SegFormer',
    'unet': 'U-Net',
    'deeplabv3plus': 'DeepLabv3+'
}

# colors = sns.color_palette('magma', n_colors=3)
colors = sns.color_palette(n_colors=3)
s2_color = 'black'
# colors = colors[1:]  # Exclude the first color for baseline

class_wise_metrics_df = pd.read_csv('../runs/s2flow_lc_sampling_steps_sweep/class_wise_metrics.csv')

for model in ['segformer', 'deeplabv3plus', 'unet']:
    model_df = class_wise_metrics_df[class_wise_metrics_df['model'] == model]
    
    # Isolate baseline (Sentinel-2)
    baseline = model_df[model_df['sampler'] == 's2'].set_index(['fold'])
    
    # Calculate difference for other samplers
    for sampler in ['realesrgan', 'euler', 'naip']:
        curr_sampler = model_df[model_df['sampler'] == sampler].set_index(['fold'])
        
        # Calculate delta for each class metric
        delta_df = curr_sampler.filter(like='class_') - baseline.filter(like='class_')
        delta_df = delta_df.reset_index()
        delta_df['sampler'] = sampler
        delta_df['model'] = model
        deltas.append(delta_df)

delta_master_df = pd.concat(deltas)

# 2. Plotting
fig, ax = plt.subplots(3, 3, figsize=(12, 9), sharex=True, sharey=True)

for i, model in enumerate(['segformer', 'deeplabv3plus', 'unet']):
    for j, metric in enumerate(['precision', 'recall', 'f1_score']):
        plot_df = delta_master_df[delta_master_df['model'] == model]
        
        # Melt for plotting
        melted = plot_df.melt(
            id_vars=['sampler', 'fold'],
            value_vars=[f'class_{k}_{metric}' for k in class_names.keys()],
            var_name='class_name',
            value_name='delta'
        )
        melted['class_name'] = melted['class_name'].apply(lambda x: class_names[int(x.split('_')[1])])
        melted['sampler_label'] = melted['sampler'].map(fancy_data_labels)

        sns.barplot(
            data=melted,
            y='class_name',
            x='delta',
            hue='sampler_label',
            palette=colors,
            ax=ax[i][j],
            errorbar='sd',
            err_kws={
                'linewidth': 3,
                'alpha': 0.8,
                'color': 'gray'
                
            }
            # 'errorbar': 'sd',
            # 'errwidth': 1.5,
            # 'errcolor': 'gray'
        )
        
        # Scientific Styling
        ax[i][j].axvline(0, color=s2_color, linewidth=0.5, linestyle='-')
        ax[i][j].set_ylabel('')
        
        
        # if i == 0:
            # ax[0][j].set_title(fancy_model_names[model], fontsize=14, fontweight='bold')
        
        # Legend handling
        if i == 0 and j == 0:
            ax[i][j].legend(title='Data Source', loc='upper left', bbox_to_anchor=(0, 1), fontsize=9)
        else:
            ax[i][j].get_legend().remove()
    
        ax[i][j].set_xlabel('$\\Delta$ ' + fancy_metric_names[metric], fontsize=14)
        ax[i][j].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))
        ax[i][j].grid(True, axis='x', linestyle='--', alpha=0.5)
        # use minor gridlines as well
        ax[i][j].xaxis.minorticks_on()
        ax[i, j].grid(True, axis='x', which='minor', linestyle='--', alpha=0.1)

    # show model name on right side of the rightmost plot
    ax[i][-1].text(1.05, 0.5, fancy_model_names[model], transform=ax[i][-1].transAxes, fontsize=14, fontweight='bold', va='center')

# center x = 0 lines
running_xmin, running_xmax = float('inf'), float('-inf')
for axis in ax.flatten():
    xmin, xmax = axis.get_xlim()
    running_xmin = min(running_xmin, xmin)
    running_xmax = max(running_xmax, xmax)

# use symmetric x limits for all plots
if abs(running_xmin) > abs(running_xmax):
    running_xmax = abs(running_xmin)
else:
    running_xmin = -abs(running_xmax)
for axis in ax.flatten():
    axis.set_xlim(running_xmin, running_xmax)

# fig.supxlabel('Change in Accuracy Compared to Sentinel-2', fontsize=14, y=0.02)

plt.tight_layout()
plt.show()

out_path = Path('../figures/sampling_lc_delta_comparison_2')
out_path.mkdir(parents=True, exist_ok=True)
for filetype in ['png', 'pdf']:
    fig.savefig(out_path / f'sampling_lc_delta_heatmap.{filetype}', dpi=300 if filetype=='png' else None, bbox_inches='tight')

In [ ]:

f1_delta = delta_master_df.melt(
    id_vars=['model', 'sampler', 'fold'],
    value_vars=[f'class_{class_id}_f1_score' for class_id in class_names],
    var_name='class_id',
    value_name='delta_f1',
)
f1_delta['class_name'] = f1_delta['class_id'].map(
    lambda value: class_names[int(value.split('_')[1])]
)
f1_delta_mean = f1_delta.groupby(
    ['class_name', 'model', 'sampler'], observed=True
)['delta_f1'].mean().unstack(['model', 'sampler'])

classifier_order = ['segformer', 'deeplabv3plus', 'unet']
f1_delta_mean = f1_delta_mean.loc[
    list(class_names.values()),
    [(classifier, 'euler') for classifier in classifier_order],
].T
f1_delta_mean.index = ['SegFormer', 'DeepLabV3+', 'U-Net']

color_limit = max(f1_delta_mean.abs().to_numpy().max(), 0.01)
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    f1_delta_mean,
    ax=ax,
    cmap='coolwarm',
    center=0,
    vmin=-color_limit,
    vmax=color_limit,
    annot=True,
    fmt='.2%',
    linewidths=0.5,
    cbar_kws={'label': '$\\Delta$ F1 Score'},
)
# ax.set_title('Class-wise F1 change for Euler (T=1) vs. Sentinel-2')
ax.set_xlabel('')
ax.set_ylabel('')
# ax.tick_params(axis='x',)
# ax.tick_params(axis='y', rotation=90)
ax.collections[0].colorbar.ax.yaxis.set_major_formatter(
    plt.FuncFormatter(lambda value, _: f'{value:.1%}')
)
fig.tight_layout()

out_path = Path('../figures/sampling_lc_delta_heatmap/')
out_path.mkdir(parents=True, exist_ok=True)
for filetype in ['png', 'pdf']:
    fig.savefig(out_path / f'sampling_lc_delta_heatmap.{filetype}', dpi=300 if filetype=='png' else None, bbox_inches='tight')